# Understanding TF-IDF: The Math Behind Word Importance

In Natural Language Processing, text must be converted into numerical representations before a machine learning model can process it. While Bag-of-Words (BoW) simply counts word frequencies, **TF-IDF (Term Frequency-Inverse Document Frequency)** weighs words based on their actual informational value.

TF-IDF operates on a simple philosophy: **A word is important to a document if it appears frequently in that specific document, but rarely across the entire dataset.**

---

## 1. Component Comparison: TF vs. IDF

To understand how TF-IDF determines "importance," we must break it down into its two distinct engines. They work in opposite directions to find the perfect balance.

| Feature | Term Frequency (TF) | Inverse Document Frequency (IDF) |
| :--- | :--- | :--- |
| **Scope** | **Local** (Document level) | **Global** (Corpus/Dataset level) |
| **Question it Asks** | "How often is this word used *here*?" | "How common is this word *everywhere else*?" |
| **Informative Value** | Measures **Relevance** to the current text. | Measures **Specificity** or "Surprise" factor. |
| **Behavior** | Increases as the word count in the document increases. | Decreases as the word appears in more documents. |



---

## 2. Mathematical Representation

Here is the formal mathematics behind the vectorizer. 

### A. Term Frequency (TF)
The raw count of a term $t$ in a document $d$, divided by the total number of terms in that document to normalize the length (so long documents don't naturally dominate short ones).

$$TF(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}$$
* $f_{t,d}$: The frequency (count) of term $t$ in document $d$.
* $\sum f_{t',d}$: The total number of all words in document $d$.

### B. Inverse Document Frequency (IDF)
The logarithm of the total number of documents $N$ divided by the number of documents that contain the term $t$. 

$$IDF(t, D) = \log\left(\frac{N}{|\{d \in D : t \in d\}|}\right)$$
* $N$: Total number of documents in the corpus $D$.
* $|\{d \in D : t \in d\}|$: The number of documents where the term $t$ appears. 
*(Note: In practice, libraries like Scikit-Learn add +1 to the denominator to prevent division by zero).*

### C. The Final Weight (TF-IDF)
The components are multiplied together.

$$TF\text{-}IDF(t, d, D) = TF(t, d) \times IDF(t, D)$$

---

## 3. Scenarios: How TF-IDF Determines Importance

Let's look at four scenarios across a massive dataset of 1,000,000 articles to see how the multiplication of these two metrics dictates "importance."

### Scenario 1: The "Stopword" (High TF, Low IDF)
* **Word:** "the"
* **Situation:** Appears 50 times in Doc A. Appears in all 1,000,000 documents.
* **Math:** High TF $\times$ $\log(1,000,000 / 1,000,000)$
* **Result:** High TF $\times$ $0$ = **0**
* **Interpretation:** Even though it is heavily used locally, it offers zero global informational value. The model mutes it.

### Scenario 2: The "Keyword" (High TF, High IDF)
* **Word:** "TensorFlow"
* **Situation:** Appears 15 times in Doc A. Appears in only 500 out of 1,000,000 documents.
* **Math:** High TF $\times$ $\log(1,000,000 / 500)$
* **Result:** High TF $\times$ $3.3$ = **High Score**
* **Interpretation:** The word is heavily used locally AND rare globally. This is the "Sweet Spot." The model identifies this as the core topic of the document.



### Scenario 3: The "Passing Mention" (Low TF, High IDF)
* **Word:** "Keras"
* **Situation:** Appears 1 time in Doc A. Appears in 500 documents globally.
* **Math:** Low TF $\times$ $\log(1,000,000 / 500)$
* **Result:** Low TF $\times$ $3.3$ = **Moderate Score**
* **Interpretation:** The word is rare globally, which is good, but it's barely mentioned in this document. It gets a decent weight, but won't be the defining keyword of the text.

### Scenario 4: The "Irrelevant Common" (Low TF, Low IDF)
* **Word:** "is"
* **Situation:** Appears 1 time in a very short Doc A. Appears in 900,000 documents globally.
* **Math:** Low TF $\times$ $\log(1,000,000 / 900,000)$
* **Result:** Low TF $\times$ $0.04$ = **Near Zero**
* **Interpretation:** It isn't frequent locally, and it is extremely common globally. The model safely ignores it.

---

## 4. Concrete Tracing Example

Let's trace the math on a mini-corpus of 3 documents ($N = 3$):
1. **Doc 1:** "data science is fun" (4 words)
2. **Doc 2:** "data is the new oil" (5 words)
3. **Doc 3:** "machine learning is fun" (4 words)

Let's evaluate the words **"data"** and **"is"** for **Doc 1**.

**Evaluating "data" in Doc 1:**
* $TF = 1 / 4 = 0.25$
* $IDF = \log_{10}(3 / 2) = \log_{10}(1.5) \approx 0.176$
* $TF\text{-}IDF = 0.25 \times 0.176 = \mathbf{0.044}$

**Evaluating "is" in Doc 1:**
* $TF = 1 / 4 = 0.25$
* $IDF = \log_{10}(3 / 3) = \log_{10}(1) = \mathbf{0}$
* $TF\text{-}IDF = 0.25 \times 0 = \mathbf{0}$

**Conclusion:** The algorithm successfully learned that **"data"** carries more informative weight for Doc 1 than the word **"is"**, purely through the mathematical interaction of local frequency and global rarity.

In [2]:
import nltk
import string
from nltk.tokenize import word_tokenize
nltk.download('punkt')
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk.stem import WordNetLemmatizer
nltk.download("wordnet")

[nltk_data] Downloading package punkt to /home/tamer/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/tamer/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/tamer/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [1]:
reviews = [ 
"I loved the movie. It was amazing!", 
"The movie was okay.", 
"I hated the movie. It was boring." 
]

In [3]:
lemmatizer = WordNetLemmatizer()

In [4]:
def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in string.punctuation]
    stop_words = stopwords.words('english')
    tokens = [word for word in tokens if word not in stop_words]
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(lemmatized_tokens)

In [5]:
cleaned_reviews = [preprocess(review) for review in reviews]
print(cleaned_reviews)

['loved movie amazing', 'movie okay', 'hated movie boring']


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
# The Tf-IDF create the vocabulary and sort it in alphabetical order.

tfidf_matrix = vectorizer.fit_transform(cleaned_reviews)
print(tfidf_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 8 stored elements and shape (3, 6)>
  Coords	Values
  (0, 0)	0.652490884512534
  (0, 4)	0.3853716274664007
  (0, 3)	0.652490884512534
  (1, 5)	0.8610369959439764
  (1, 4)	0.5085423203783267
  (2, 1)	0.652490884512534
  (2, 2)	0.652490884512534
  (2, 4)	0.3853716274664007


In [10]:
print(tfidf_matrix.toarray())

[[0.65249088 0.         0.         0.65249088 0.38537163 0.        ]
 [0.         0.         0.         0.         0.50854232 0.861037  ]
 [0.         0.65249088 0.65249088 0.         0.38537163 0.        ]]


In [9]:
vectorizer.get_feature_names_out()

array(['amazing', 'boring', 'hated', 'loved', 'movie', 'okay'],
      dtype=object)

# Implement the TF-IDF Vectorizer from scratch

In [11]:
cleaned_reviews

['loved movie amazing', 'movie okay', 'hated movie boring']

In [27]:
l2 = []

In [28]:
l2.extend(cleaned_reviews[0].split())


In [29]:
l2

['loved', 'movie', 'amazing']

In [21]:
all_words = []
for review in cleaned_reviews:
    all_words.extend(review.split())

In [22]:
all_words

['loved', 'movie', 'amazing', 'movie', 'okay', 'hated', 'movie', 'boring']

In [30]:
import pandas as pd
# Convert the cleaned_reviews into a DataFrame to get the Unique Words
df = pd.DataFrame(all_words, columns=['all_words'])
df

,all_words
0,loved
1,movie
2,amazing
3,movie
4,okay
5,hated
6,movie
7,boring


Get the Vocab

In [32]:
vocabulary = df["all_words"].unique()
vocabulary

array(['loved', 'movie', 'amazing', 'okay', 'hated', 'boring'],
      dtype=object)

In [33]:
# Sorting the Vocabulary in Alphabetical Order
vocabulary.sort()



In [34]:
vocabulary

array(['amazing', 'boring', 'hated', 'loved', 'movie', 'okay'],
      dtype=object)

In [35]:
import numpy as np


Caculate the IDF

In [41]:
# calculate the  number of documents that containing the word 
document_count = {}
for i in vocabulary:
    count = 0
    for review in cleaned_reviews:
        if i in review.split():
            count += 1
            # print(f"{i} is in {review}")
            # print(f"{i} is in {count} documents")
    document_count[i] = count

print(document_count)


{'amazing': 1, 'boring': 1, 'hated': 1, 'loved': 1, 'movie': 3, 'okay': 1}


In [37]:
count

1

In [44]:
idf_values = {}
for word, count in document_count.items():
    print(f"{word} is in {count} documents")
    idf = np.log(len(cleaned_reviews) / count)
    print(f"IDF for {word}: {idf}")
    idf_values[word] = idf
    

amazing is in 1 documents
IDF for amazing: 1.0986122886681098
boring is in 1 documents
IDF for boring: 1.0986122886681098
hated is in 1 documents
IDF for hated: 1.0986122886681098
loved is in 1 documents
IDF for loved: 1.0986122886681098
movie is in 3 documents
IDF for movie: 0.0
okay is in 1 documents
IDF for okay: 1.0986122886681098


In [45]:
idf_values

{'amazing': 1.0986122886681098,
 'boring': 1.0986122886681098,
 'hated': 1.0986122886681098,
 'loved': 1.0986122886681098,
 'movie': 0.0,
 'okay': 1.0986122886681098}

In [46]:
# Loop through the cleaned_reviews and calculate the TF-IDF for each word in each document
tf_idf_values = []
for review in cleaned_reviews:
    tf_idf_review = {}
    for word in review.split():
        tf = review.split().count(word) / len(review.split())
        idf = idf_values[word]
        tf_idf = tf * idf
        tf_idf_review[word] = tf_idf
    tf_idf_values.append(tf_idf_review)

In [47]:
tf_idf_values

[{'loved': 0.3662040962227032, 'movie': 0.0, 'amazing': 0.3662040962227032},
 {'movie': 0.0, 'okay': 0.5493061443340549},
 {'hated': 0.3662040962227032, 'movie': 0.0, 'boring': 0.3662040962227032}]

In [48]:
# normalize the TF-IDF values
normalized_tf_idf_values = []
for review in tf_idf_values:
    normalized_review = {}
    for word, tf_idf in review.items():
        normalized_tf_idf = tf_idf / np.linalg.norm(list(review.values()))
        normalized_review[word] = normalized_tf_idf
    normalized_tf_idf_values.append(normalized_review)
normalized_tf_idf_values

[{'loved': 0.7071067811865475, 'movie': 0.0, 'amazing': 0.7071067811865475},
 {'movie': 0.0, 'okay': 1.0},
 {'hated': 0.7071067811865475, 'movie': 0.0, 'boring': 0.7071067811865475}]

In [49]:
# Create the TF-IDF matrix
tf_idf_matrix = np.zeros((len(cleaned_reviews), len(vocabulary)))
for i, review in enumerate(cleaned_reviews):
    for j, word in enumerate(vocabulary):
        if word in review.split():
            tf_idf_matrix[i][j] = tf_idf_values[i][word]   
tf_idf_matrix

array([[0.3662041 , 0.        , 0.        , 0.3662041 , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.54930614],
       [0.        , 0.3662041 , 0.3662041 , 0.        , 0.        ,
        0.        ]])

In [50]:
# Create a DataFrame to display the TF-IDF values
tf_idf_df = pd.DataFrame(normalized_tf_idf_values)
tf_idf_df.fillna(0, inplace=True)
tf_idf_df.columns = vocabulary
tf_idf_df   


,amazing,boring,hated,loved,movie,okay
0,0.707107,0.0,0.707107,0.0,0.000000,0.000000
1,0.000000,0.0,0.000000,1.0,0.000000,0.000000
2,0.000000,0.0,0.000000,0.0,0.707107,0.707107


In [51]:
# Visualize the heatmap of the TF-IDF values
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
sns.heatmap(tf_idf_df, annot=True, cmap="YlGnBu", xticklabels=vocabulary, yticklabels=[f"Review {i+1}" for i in range(len(cleaned_reviews))])
plt.title("TF-IDF Heatmap")
plt.xlabel("Words")
plt.ylabel("Reviews")
plt.show()

ModuleNotFoundError: No module named 'seaborn'